# Solution: Simple PID Controller


In [12]:
# Import standard libraries
import math
from pathlib import Path
import time

# Import third-party libraries
from IPython.display import clear_output
import mujoco
import mujoco.viewer
import numpy as np

In [13]:
# Settings
MJCF_PATH = Path("../../mechanical/bala-c-plus-simplified/bala-c-plus-simplified.xml")
MOTOR_SPEED_LIMIT = 1.0    # Max motor speed in each direction
PRINT_EVERY = 50           # Number of sim loop iterations before printing sensor readings

# Actuator names (from MJCF file)
LEFT_MOTOR = "left_motor"
RIGHT_MOTOR = "right_motor"

# Sensor names (from MJCF file)
IMU_ACCEL = "imu_accel"
IMU_GYRO = "imu_gyro"
IMU_ORIENTATION = "imu_orientation"

In [14]:
def clamp(x):
    """Limit motor speed and direction to a minimum and maximum"""
    return max(-MOTOR_SPEED_LIMIT, min(MOTOR_SPEED_LIMIT, x))

In [15]:
# Load model into MuJoCo
model = mujoco.MjModel.from_xml_path(str(MJCF_PATH))

# Use model to get the simulation state
data  = mujoco.MjData(model)

In [16]:
# Get ID of actuators from MJCF names
left_motor_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, LEFT_MOTOR)
right_motor_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, RIGHT_MOTOR)

# Print IDs
print(f"Left motor ID: {left_motor_id}")
print(f"Right motor ID: {right_motor_id}")

Left motor ID: 0
Right motor ID: 1


In [33]:
# Adjust that arange to figure out where the bot balances most naturally
for deg in np.arange(11.3, 11.5, 0.01):
    th = math.radians(deg)
    mujoco.mj_resetData(model, data)
    data.qpos[3:7] = [math.cos(th/2), 0, math.sin(th/2), 0]
    data.qvel[:] = 0
    mujoco.mj_forward(model, data)
    p0 = deg
    for _ in range(100):
        data.ctrl[:] = 0
        mujoco.mj_step(model, data)
    w,x,y,z = data.sensor("imu_orientation").data
    p1 = math.degrees(math.atan2(1-2*(x*x+y*y), -2*(y*z+w*x)))
    print(f"start {deg:.2f} -> after 100 steps {p1:+.2f} (moved {abs(p1-abs(p0)):.2f})")

start 11.30 -> after 100 steps +13.46 (moved 2.16)
start 11.31 -> after 100 steps +13.26 (moved 1.95)
start 11.32 -> after 100 steps +13.07 (moved 1.75)
start 11.33 -> after 100 steps +12.87 (moved 1.54)
start 11.34 -> after 100 steps +12.67 (moved 1.33)
start 11.35 -> after 100 steps +12.48 (moved 1.13)
start 11.36 -> after 100 steps +12.28 (moved 0.92)
start 11.37 -> after 100 steps +12.08 (moved 0.71)
start 11.38 -> after 100 steps +11.88 (moved 0.50)
start 11.39 -> after 100 steps +11.69 (moved 0.30)
start 11.40 -> after 100 steps +11.49 (moved 0.09)
start 11.41 -> after 100 steps +11.29 (moved 0.12)
start 11.42 -> after 100 steps +11.09 (moved 0.33)
start 11.43 -> after 100 steps +10.90 (moved 0.53)
start 11.44 -> after 100 steps +10.70 (moved 0.74)
start 11.45 -> after 100 steps +10.50 (moved 0.95)
start 11.46 -> after 100 steps +10.31 (moved 1.15)
start 11.47 -> after 100 steps +10.11 (moved 1.36)
start 11.48 -> after 100 steps +9.91 (moved 1.57)
start 11.49 -> after 100 steps +

In [22]:
# Filter and PID coefficients (tune these)
KP = 20.0
KD = 0.5

# Trim angle: true angle the bot needs to balance at (with offset CoM)
PITCH_TRIM = math.radians(-12.5)

# When the robot has "tipped over" into an unrecoverable state
TIP_THRESHOLD = math.radians(30)

# Resets simulation data to defaults
mujoco.mj_resetData(model, data)

# Launch MuJoCo simulator and GUI
steps = 0
pitch = 0.0
tipped = False
prev_time = 0.0
with mujoco.viewer.launch_passive(model, data) as viewer:
    # Define free-look camera (control with mouse), looking at robot's back-right
    viewer.cam.type = mujoco.mjtCamera.mjCAMERA_FREE
    viewer.cam.lookat[:] = [0, 0, 0.05]
    viewer.cam.distance  = 0.8 
    viewer.cam.azimuth   = 45
    viewer.cam.elevation = -25

    # Simulation loop
    while viewer.is_running():
        step_start = time.time()

        w, x, y, z = data.sensor(IMU_ORIENTATION).data
        pitch = math.atan2(1.0 - 2.0*(x*x+y*y), -2.0*(y*z+w*x))
        pitch_rate = data.sensor(IMU_GYRO).data[0]
    
        ctrl = -(KP*(pitch - PITCH_TRIM) + KD*pitch_rate)
        ctrl = max(-1.0, min(1.0, ctrl))
        data.ctrl[left_motor_id]  = ctrl
        data.ctrl[right_motor_id] = ctrl
        mujoco.mj_step(model, data)
    
        if steps % 20 == 0:
            wv = data.sensor("left_wheel_vel").data[0]
            print(f"t={steps:4d}  pitch={math.degrees(pitch):+6.2f}  rate={pitch_rate:+6.2f}  ctrl={ctrl:+.3f}  wheel={wv:+.2f}")
        
        if abs(pitch) > math.radians(45):
            continue
            print(f"tipped at step {steps}");

        # Render the current simulation state
        viewer.sync()

        # Just print what the sensors say when the robot is upright
        viewer.sync()
        slack = model.opt.timestep - (time.time() - step_start)
        if slack > 0:
            time.sleep(slack)
        steps += 1

t=   0  pitch= +0.00  rate= +0.00  ctrl=-1.000  wheel=+0.00
t=  20  pitch=-11.16  rate= -0.61  ctrl=-0.164  wheel=-2.04
t=  40  pitch=-11.86  rate= -0.00  ctrl=-0.222  wheel=-1.28
t=  60  pitch=-11.84  rate= +0.01  ctrl=-0.234  wheel=-1.40
t=  80  pitch=-11.80  rate= +0.01  ctrl=-0.246  wheel=-1.53
t= 100  pitch=-11.77  rate= +0.01  ctrl=-0.258  wheel=-1.67
t= 120  pitch=-11.73  rate= +0.01  ctrl=-0.271  wheel=-1.82
t= 140  pitch=-11.69  rate= +0.01  ctrl=-0.285  wheel=-1.97
t= 160  pitch=-11.65  rate= +0.01  ctrl=-0.299  wheel=-2.13
t= 180  pitch=-11.61  rate= +0.01  ctrl=-0.313  wheel=-2.30
t= 200  pitch=-11.57  rate= +0.01  ctrl=-0.328  wheel=-2.47
t= 220  pitch=-11.53  rate= +0.01  ctrl=-0.344  wheel=-2.66
t= 240  pitch=-11.48  rate= +0.01  ctrl=-0.361  wheel=-2.84
t= 260  pitch=-11.43  rate= +0.01  ctrl=-0.378  wheel=-3.04
t= 280  pitch=-11.38  rate= +0.01  ctrl=-0.396  wheel=-3.25
t= 300  pitch=-11.33  rate= +0.01  ctrl=-0.415  wheel=-3.46
t= 320  pitch=-11.27  rate= +0.01  ctrl=

KeyboardInterrupt: 

In [9]:
mujoco.mj_resetData(model, data)
for deg in [-12.6, 12.6]:
    th = math.radians(deg)
    # set chassis pitch, let it sit, see which way it falls
    data.qpos[3:7] = [math.cos(th/2), 0, math.sin(th/2), 0]
    data.qvel[:] = 0
    mujoco.mj_forward(model, data)
    # step a few times with ZERO control, watch pitch drift
    for _ in range(50):
        data.ctrl[:] = 0
        mujoco.mj_step(model, data)
    w,x,y,z = data.sensor(IMU_ORIENTATION).data
    p = math.atan2(1.0-2.0*(x*x+y*y), -2.0*(y*z+w*x))
    print(f"start {deg:+.1f}deg -> after 50 steps, pitch={math.degrees(p):+.2f}")

start -12.6deg -> after 50 steps, pitch=+70.51
start +12.6deg -> after 50 steps, pitch=-12.53
